In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()   # because notebook is inside rfa_python/notebooks
sys.path.insert(0, str(PROJECT_ROOT))

import importlib

import rfa_model.geometry as geom
importlib.reload(geom)
from rfa_model.geometry import *
from rfa_model.io import load_field_npz
from rfa_model.fields import *
from rfa_model.collisions import *

import rfa_model.trajectories as traj_mod
import rfa_model.primary as primary_mod

importlib.reload(traj_mod)
importlib.reload(primary_mod)

from rfa_model.trajectories import *
from rfa_model.primary import *

import rfa_model.accounting as acc
importlib.reload(acc)
from rfa_model.accounting import *

import rfa_model.samplers as samplers
importlib.reload(samplers)
from rfa_model.samplers import *

import rfa_model.batches as batches
importlib.reload(batches)
from rfa_model.batches import *

import rfa_model.cascade as cascade
importlib.reload(cascade)
from rfa_model.cascade import *

In [14]:
%matplotlib widget

## Load STLs

In [15]:
stl_dir = Path("../rfa stl")

meshes, sample_parts = load_and_align_sample_assembly(
    stl_dir,
    alpha_deg=0.0,
)

frame_meshes, frame_parts, frame_info = load_and_align_grid_frames(
    stl_dir,
    verbose=True,
)

sample_y_bounds, sample_z_bounds = sample_bounds(meshes)

Grid frame alignment
  fitted center: [-0.00113118 -0.00140144 -0.00109564]
  fitted radius: 0.05887084242766213
  axis before alignment: [ 9.99990913e-01  4.23261006e-03 -5.08542373e-04]
  R_g1: 0.04519042252564098
  R_g2: 0.05792646490184952
  R_g3: 0.07107616160072151


In [16]:
collision_meshes_primary = build_collision_mesh_dict(meshes, frame_meshes, include_sample=True)
collision_meshes_emit    = build_collision_mesh_dict(meshes, frame_meshes, include_sample=False)

In [17]:
print("rod" in [k.lower() for k in collision_meshes_emit.keys()])
print(collision_meshes_emit.keys())

True
dict_keys(['holder', 'receiver', 'rod', 'g1_low_frame', 'g1_upper_frame', 'g2_low_frame', 'g2_upper_frame', 'g3_low_frame', 'g3_upper_frame'])


In [19]:
rod_mesh = collision_meshes_emit["rod"]  # or whatever the actual key is
print(rod_mesh.bounds)
print(rod_mesh.extents)  # size in x/y/z

[[-0.01281619 -0.01269927 -0.16704621]
 [ 0.00786809  0.0127      0.01075379]]
[0.02068428 0.02539927 0.1778    ]


In [22]:
def rod_cross_section_at_radius(rod_mesh, R, tol=0.002):
    verts = rod_mesh.vertices
    r = np.linalg.norm(verts, axis=1)
    mask = np.abs(r - R) < tol
    pts = verts[mask]
    if len(pts) == 0:
        print(f"R={R}: no vertices near this radius (rod may not reach here)")
        return
    print(f"R={R}: x range {pts[:,0].min():.4f} to {pts[:,0].max():.4f}, "
          f"y range {pts[:,1].min():.4f} to {pts[:,1].max():.4f}, "
          f"center=({pts[:,0].mean():.4f}, {pts[:,1].mean():.4f})")

for R in [field["R_g1"], field["R_g2"], field["R_g3"], field["R_col"]]:
    rod_cross_section_at_radius(collision_meshes_emit["rod"], R)

R=0.04519042252564098: no vertices near this radius (rod may not reach here)
R=0.05792646490184952: x range -0.0078 to -0.0001, y range -0.0033 to 0.0034, center=(-0.0027, 0.0001)
R=0.07107616160072151: x range -0.0078 to 0.0078, y range -0.0079 to 0.0079, center=(0.0005, 0.0000)
R=0.08255: no vertices near this radius (rod may not reach here)


## Load field

In [20]:
field = load_field_npz("../results/fields/rfa_field_sample_50_g2g3_0_collector50_0p25mm.npz")

print(field.keys())
print("V shape:", field["V"].shape)
print("h:", field["h"])
print("voltages:", field["voltages"])

dict_keys(['x', 'y', 'z', 'h', 'V', 'Ex', 'Ey', 'Ez', 'R_g1', 'R_g2', 'R_g3', 'R_col', 'fixed', 'update_region', 'Vfix', 'owner', 'voltages', 'Vs', 'Vr', 'Vg1', 'Vg2', 'Vg3', 'Vc', 'Vdt'])
V shape: (665, 665, 665)
h: 0.00025
voltages: {'Vs': 50.0, 'Vr': 0.0, 'Vg1': 0.0, 'Vg2': 0.0, 'Vg3': 0.0, 'Vc': 50.0, 'Vdt': 0.0}


In [21]:
Ex_interp, Ey_interp, Ez_interp = build_field_interpolators(field)
Phi_interp = build_potential_interpolator(field)

p_test = np.array([0.001, 0.0, 0.0])

print("E =", E_at_point(p_test, Ex_interp, Ey_interp, Ez_interp))
print("Phi =", potential_at_point(p_test, Phi_interp))
print("speed 500 eV =", speed_from_energy_eV(500))

E = [3153.40859066   14.52731429   -6.77108569]
Phi = 46.86360372407512
speed 500 eV = 13262051.164024979


In [6]:
voltages = field["voltages"]
field = attach_default_owner_name_map(field)

In [7]:
field["owner_name_map"]

{0: 'free',
 1: 'sample',
 2: 'holder',
 3: 'receiver',
 4: 'rod',
 5: 'g1frame',
 6: 'g2frame',
 7: 'g3frame',
 8: 'drifttube',
 9: 'g1_shell',
 10: 'g2_shell',
 11: 'g3_shell',
 12: 'collector_shell'}

## Build intersectors

In [8]:
collision_meshes_primary = build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=True,
)

collision_mesh_primary, face_owner_primary, intersector_primary = build_stl_intersector(
    collision_meshes_primary
)

stl_boxes_primary = build_stl_bounding_boxes(
    collision_meshes_primary,
    padding=1.0e-3,
)

collision_meshes_emit = build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=False,
)

collision_mesh_emit, face_owner_emit, intersector_emit = build_stl_intersector(
    collision_meshes_emit
)

stl_boxes_emit = build_stl_bounding_boxes(
    collision_meshes_emit,
    padding=1.0e-3,
)

## Test primary beam

In [20]:
y0, z0 = sample_center_from_bounds(sample_y_bounds, sample_z_bounds)

rng = np.random.default_rng(123)

x_start_primary = 0.75 * field["h"]

p0s, v0s, K0s, Phi0s = make_primary_beam_near_sample(
    N=20,
    E0_eV=500,
    field=field,
    Phi_interp=Phi_interp,
    x_start=x_start_primary,
    y0=y0,
    z0=z0,
    beam_sigma=150e-6,
    energy_spread_eV=0.0,
    angular_sigma_deg=0.0,
    sample_voltage=voltages["Vs"],
    rng=rng,
)


for i in range(20):
    res = fly_primary_to_sample(
        p0=p0s[i],
        v0=v0s[i],
        field=field,
        Ex_interp=Ex_interp,
        Ey_interp=Ey_interp,
        Ez_interp=Ez_interp,
        Phi_interp=Phi_interp,
        intersector=intersector_primary,
        face_owner=face_owner_primary,
        collision_mesh=collision_mesh_primary,
        stl_boxes=stl_boxes_primary,
        sample_y_bounds=sample_y_bounds,
        sample_z_bounds=sample_z_bounds,
    )

    print(i, res["reason"], res["hit_info"].get("kind"), res["hit_info"].get("KE_hit_eV"))
    print("  location:", res["hit_info"].get("location"))

0 hit_sample sample_voxel 500.0000250949711
  location: [ 0.000225   -0.00026837 -0.00033112]
1 hit_sample sample_voxel 500.0000249299254
  location: [ 0.000225   -0.00017517  0.00012419]
2 hit_sample sample_voxel 500.00002400647963
  location: [2.25000000e-04 7.31889036e-05 2.31244559e-04]
3 hit_sample sample_voxel 500.00002460716297
  location: [ 2.25000000e-04 -9.09037227e-05  1.69021019e-04]
4 hit_sample sample_voxel 500.0000241726189
  location: [2.25000000e-04 1.80347493e-05 1.13215447e-04]
5 hit_sample sample_voxel 500.00002433092874
  location: [ 2.25000000e-04 -3.34343169e-05 -2.18966840e-05]
6 hit_sample sample_voxel 500.00002510431284
  location: [ 0.000225   -0.00021547  0.00019229]
7 hit_sample sample_voxel 500.00002439924424
  location: [ 2.25000000e-04 -3.87070526e-05  1.61104593e-04]
8 hit_sample sample_voxel 500.0000248849938
  location: [ 2.25000000e-04 -1.67489203e-04  5.88931267e-05]
9 hit_sample sample_voxel 500.0000248754264
  location: [ 2.25000000e-04 -1.6835825

## Load samplers

In [9]:
model_dir = Path("../data/jmonsel")
bronstein_dir = Path("../data/bronstein")

yield_models, energy_models, theta_models = load_default_surface_models(
    model_dir=model_dir,
    bronstein_dir=bronstein_dir,
)

## Test one primary with emission from the sample

In [24]:
primary_results = []
emitted_results_all = []

rng = np.random.default_rng(123)

N_test = 20

grid_transparency = {
    "g1_shell": 0.85,
    "g2_shell": 0.85,
    "g3_shell": 0.85,
}

for i in range(N_test):
    primary_res_i, emitted_i = run_one_primary_with_model_emission(
        p_primary=p0s[i % len(p0s)],
        v_primary=v0s[i % len(v0s)],
        field=field,
        Ex_interp=Ex_interp,
        Ey_interp=Ey_interp,
        Ez_interp=Ez_interp,
        Phi_interp=Phi_interp,

        intersector_primary=intersector_primary,
        face_owner_primary=face_owner_primary,
        collision_mesh_primary=collision_mesh_primary,
        stl_boxes_primary=stl_boxes_primary,

        intersector_emit=intersector_emit,
        face_owner_emit=face_owner_emit,
        collision_mesh_emit=collision_mesh_emit,
        stl_boxes_emit=stl_boxes_emit,

        grid_transparency=grid_transparency,

        yield_models=yield_models,
        energy_models=energy_models,
        theta_models=theta_models,
        voltages=voltages,
        SEY_mult=1.0,
        rng=rng,

        sample_y_bounds=sample_y_bounds,
        sample_z_bounds=sample_z_bounds,
    )

    primary_results.append(primary_res_i)
    emitted_results_all.append(emitted_i)

acct_many = summarize_many_first_generation(
    primary_results=primary_results,
    emitted_results_all=emitted_results_all,
    owner_name_map=field["owner_name_map"],
)

print_many_first_generation_summary(acct_many)

acct_many["df_emit"].head()

Many-primary first-generation accounting
---------------------------------------
N primary:             20
N primary hit sample:  20
Hit-sample fraction:   1.00000

N emitted total:       30
N SE:                  21
N BSE:                 9
N quantum refl.:       0

Per primary emitted:   1.50000
Per primary SE:        1.05000
Per primary BSE:       0.45000

Terminal electrodes:
terminal_electrode
sample        0
holder        0
receiver      0
rod           1
grid1         2
grid2         1
grid3         4
collector    22
drifttube     0
escaped       0
unknown       0
Name: count, dtype: int64

Terminal owners:
terminal_owner
collector_shell    22
g3_shell            4
g1frame             1
g2_shell            1
g1_shell            1
rod                 1
Name: count, dtype: int64

Emission kinds:
emission_kind
SE     21
BSE     9
Name: count, dtype: int64


,primary_index,electron_index,emission_kind,E_emit_eV,primary_E_inc_eV,primary_cos_theta,reason,terminal_owner,terminal_electrode,owner_id,KE_hit_eV,steps,x_hit,y_hit,z_hit
0,0,0,BSE,68.773578,500.000025,1.0,hit_fixed,collector_shell,collector,12,115.693467,779,0.037341,-0.033788,0.063752
1,1,0,BSE,394.990314,500.000025,1.0,hit_fixed,collector_shell,collector,12,442.512898,396,0.024294,0.077251,0.008971
2,1,1,SE,26.684137,500.000025,1.0,hit_fixed,collector_shell,collector,12,73.917447,1239,0.072791,0.031269,0.018507
3,2,0,SE,32.071899,500.000024,1.0,hit_fixed,collector_shell,collector,12,80.528677,1126,0.071183,-0.038279,0.011932
4,2,1,SE,31.990441,500.000024,1.0,hit_fixed,collector_shell,collector,12,79.572477,1123,0.059426,0.024196,0.050283


In [26]:
df_grid_events = grid_events_to_dataframe_many(
    emitted_results_all,
    owner_name_map=field["owner_name_map"],
)

df_grid_events.head()

,primary_index,electron_index,event_index,event_type,owner,electrode,step,x,y,z
0,0,0,0,transmit_fixed_voxel,g1_shell,grid1,453,0.020771,-0.018676,0.034884
1,0,0,1,transmit_fixed_voxel,g2_shell,grid2,570,0.026445,-0.023854,0.044791
2,0,0,2,transmit_fixed_voxel,g3_shell,grid3,693,0.032443,-0.029328,0.055262
3,1,0,0,transmit_fixed_voxel,g1_shell,grid1,224,0.013611,0.042349,0.004979
4,1,0,1,transmit_fixed_voxel,g2_shell,grid2,283,0.017253,0.054251,0.006338


In [27]:
df_grid_events["electrode"].value_counts()

electrode
grid1    28
grid2    26
grid3    22
Name: count, dtype: int64

In [28]:
acct_many["electrode_counts"][["grid1", "grid2", "grid3"]]

terminal_electrode
grid1    2
grid2    1
grid3    4
Name: count, dtype: int64

## Batch

In [31]:
batch20 = run_first_generation_batch_serial(
    N_primary=20,
    E0_eV=500,
    field=field,
    Phi_interp=Phi_interp,
    Ex_interp=Ex_interp,
    Ey_interp=Ey_interp,
    Ez_interp=Ez_interp,

    intersector_primary=intersector_primary,
    face_owner_primary=face_owner_primary,
    collision_mesh_primary=collision_mesh_primary,
    stl_boxes_primary=stl_boxes_primary,

    intersector_emit=intersector_emit,
    face_owner_emit=face_owner_emit,
    collision_mesh_emit=collision_mesh_emit,
    stl_boxes_emit=stl_boxes_emit,

    grid_transparency=grid_transparency,

    yield_models=yield_models,
    energy_models=energy_models,
    theta_models=theta_models,
    voltages=voltages,
    SEY_mult=1.0,

    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,

    beam_sigma=150e-6,
    seed=123,
    progress_every=5,
)

print_batch_summary(batch20)

5/20 primaries complete
10/20 primaries complete
15/20 primaries complete
20/20 primaries complete
First-generation batch summary
------------------------------
N primary:                 20
N primary hit sample:      20
Hit-sample fraction:       1.00000

N emitted total:           26
N SE:                      17
N BSE:                     9
N quantum refl.:           0

Per primary emitted total: 1.30000
Per primary SE:            0.85000
Per primary BSE:           0.45000

Runtime:                   94.47 s
Runtime per primary:       4.7236 s

Terminal electrodes:
terminal_electrode
sample        0
holder        0
receiver      0
rod           1
grid1         1
grid2         1
grid3         4
collector    19
drifttube     0
escaped       0
unknown       0
Name: count, dtype: int64

Terminal owners:
terminal_owner
collector_shell    19
g3_shell            4
g2_shell            1
g1_shell            1
rod                 1
Name: count, dtype: int64

Emission kinds:
emission_kind
SE  

In [32]:
batch20["df_emit"].head()

,primary_index,electron_index,emission_kind,E_emit_eV,primary_E_inc_eV,primary_cos_theta,reason,terminal_owner,terminal_electrode,owner_id,KE_hit_eV,steps,x_hit,y_hit,z_hit
0,1,0,BSE,52.870008,500.000025,1.0,hit_fixed,collector_shell,collector,12,99.533048,882,0.059275,0.039968,0.038937
1,1,1,SE,5.990981,500.000025,1.0,hit_fixed,collector_shell,collector,12,52.933910,2518,0.066790,-0.021834,-0.041759
2,2,0,SE,1.201346,500.000024,1.0,hit_fixed,collector_shell,collector,12,47.832554,5401,0.054254,-0.020178,0.057548
3,4,0,SE,0.604176,500.000024,1.0,hit_fixed,collector_shell,collector,12,47.750082,7619,0.078811,0.001316,-0.020723
4,4,1,SE,6.980049,500.000024,1.0,hit_fixed,collector_shell,collector,12,54.773017,2348,0.024764,0.070568,-0.032516


In [33]:
batch20["df_grid_events"].head()

,primary_index,electron_index,event_index,event_type,owner,electrode,step,x,y,z
0,1,0,0,transmit_fixed_voxel,g1_shell,grid1,513,0.032647,0.021759,0.021329
1,1,0,1,transmit_fixed_voxel,g2_shell,grid2,649,0.041805,0.028026,0.027389
2,1,0,2,transmit_fixed_voxel,g3_shell,grid3,788,0.051421,0.034608,0.033752
3,1,1,0,transmit_fixed_voxel,g1_shell,grid1,1528,0.036728,-0.011995,-0.022752
4,1,1,1,transmit_fixed_voxel,g2_shell,grid2,1929,0.047000,-0.015356,-0.029256


In [34]:
paths = save_batch_tables(
    batch20,
    out_dir="../results",
    prefix="firstgen_500eV_N20_T09",
)

paths

{'df_primary': WindowsPath('../results/firstgen_500eV_N20_T09_primary.csv'),
 'df_emit': WindowsPath('../results/firstgen_500eV_N20_T09_emit.csv'),
 'df_grid_events': WindowsPath('../results/firstgen_500eV_N20_T09_grid_events.csv'),
 'summary': WindowsPath('../results/firstgen_500eV_N20_T09_summary.csv')}

In [37]:
batch20 = add_step_diagnostics(batch20)
print_step_diagnostics(batch20)

Step diagnostics
----------------
N emitted:     26
steps mean:    1655.2
steps median:  984.0
steps p90:     2625.5
steps p95:     4734.0
steps p99:     7064.5
steps max:     7619

Slowest emitted electrons:
    primary_index  electron_index emission_kind  E_emit_eV   terminal_owner  \
3               4               0            SE   0.604176  collector_shell   
2               2               0            SE   1.201346  collector_shell   
8               9               1            SE   4.456340         g3_shell   
1               1               1            SE   5.990981  collector_shell   
17             13               1            SE   6.208274  collector_shell   
7               9               0            SE   3.785615         g2_shell   
4               4               1            SE   6.980049  collector_shell   
18             14               0            SE   1.356772              rod   
20             15               0            SE  14.309318  collector_shell   
1

## Parallel batch

In [46]:
batch20_parallel = run_first_generation_batch_parallel(
    N_primary=20,
    E0_eV=500,
    field=field,
    Phi_interp=Phi_interp,
    Ex_interp=Ex_interp,
    Ey_interp=Ey_interp,
    Ez_interp=Ez_interp,

    intersector_primary=intersector_primary,
    face_owner_primary=face_owner_primary,
    collision_mesh_primary=collision_mesh_primary,
    stl_boxes_primary=stl_boxes_primary,

    intersector_emit=intersector_emit,
    face_owner_emit=face_owner_emit,
    collision_mesh_emit=collision_mesh_emit,
    stl_boxes_emit=stl_boxes_emit,

    grid_transparency=grid_transparency,

    yield_models=yield_models,
    energy_models=energy_models,
    theta_models=theta_models,
    voltages=voltages,
    SEY_mult=1.0,

    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,

    beam_sigma=150e-6,
    seed=123,
    n_jobs=4,
    chunk_size=5,
    verbose=10,

    emitted_max_step_fraction_of_h=0.75,
    emitted_dt_max=5.0e-11,
)

print_batch_summary(batch20_parallel)
print_step_diagnostics(batch20_parallel)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    4.8s
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    6.9s remaining:    6.9s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:   13.6s finished


First-generation batch summary
------------------------------
N primary:                 20
N primary hit sample:      20
Hit-sample fraction:       1.00000

N emitted total:           29
N SE:                      21
N BSE:                     8
N quantum refl.:           0

Per primary emitted total: 1.45000
Per primary SE:            1.05000
Per primary BSE:           0.40000

Runtime:                   14.90 s
Runtime per primary:       0.7451 s

Terminal electrodes:
terminal_electrode
sample        0
holder        0
receiver      0
rod           0
grid1         1
grid2         2
grid3         3
collector    23
drifttube     0
escaped       0
unknown       0
Name: count, dtype: int64

Terminal owners:
terminal_owner
collector_shell    23
g3_shell            3
g2_shell            2
g1_shell            1
Name: count, dtype: int64

Emission kinds:
emission_kind
SE     21
BSE     8
Name: count, dtype: int64

Grid transmissions:
electrode
grid1    28
grid2    26
grid3    23
Name: count,

In [ ]:
# Conservative validation mode
emitted_max_step_fraction_of_h = 0.40
emitted_dt_max = 2.0e-11

# Exploratory / production screening mode
emitted_max_step_fraction_of_h = 0.75
emitted_dt_max = 5.0e-11

## Cascades

In [63]:
rng = np.random.default_rng(123)

primary_cas, cascade_results, cascade_log = run_one_primary_with_cascade(
    p_primary=p0s[0],
    v_primary=v0s[0],
    field=field,
    Ex_interp=Ex_interp,
    Ey_interp=Ey_interp,
    Ez_interp=Ez_interp,
    Phi_interp=Phi_interp,

    intersector_primary=intersector_primary,
    face_owner_primary=face_owner_primary,
    collision_mesh_primary=collision_mesh_primary,
    stl_boxes_primary=stl_boxes_primary,

    intersector_emit=intersector_emit,
    face_owner_emit=face_owner_emit,
    collision_mesh_emit=collision_mesh_emit,
    stl_boxes_emit=stl_boxes_emit,

    grid_transparency=grid_transparency,

    yield_models=yield_models,
    energy_models=energy_models,
    theta_models=theta_models,
    voltages=voltages,
    SEY_mult=1.0,
    rng=rng,

    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,

    max_generation=4,
    max_total_electrons=200,
    min_incident_energy_eV=0.5,

    emitted_max_step_fraction_of_h=0.75,
    emitted_dt_max=5.0e-11,
)

df_cascade = cascade_results_to_dataframe(
    cascade_results,
    owner_name_map=field["owner_name_map"],
)

df_log = cascade_log_to_dataframe(cascade_log)

print("primary:", primary_cas["reason"], primary_cas["hit_info"].get("KE_hit_eV"))
print("N cascade tracked:", len(cascade_results))

df_cascade.head(20)

primary: hit_sample 500.0000250949711
N cascade tracked: 1


,electron_id,parent_id,generation,source_owner,source_electrode,source_Einc_eV,emission_kind,E_emit_eV,reason,terminal_owner,terminal_electrode,owner_id,KE_hit_eV,steps,x_hit,y_hit,z_hit,primary_index
0,0,-1,1,sample,sample,500.000025,BSE,68.773578,hit_fixed,collector_shell,collector,12,117.418548,326,0.037335,-0.034129,0.064402,None


In [49]:
df_cascade["generation"].value_counts().sort_index()

generation
1    1
Name: count, dtype: int64

In [50]:
df_cascade["emission_kind"].value_counts()

emission_kind
BSE    1
Name: count, dtype: int64

In [51]:
df_cascade["terminal_electrode"].value_counts()

terminal_electrode
collector    1
Name: count, dtype: int64

In [52]:
df_cascade[[
    "electron_id",
    "parent_id",
    "generation",
    "source_owner",
    "source_electrode",
    "emission_kind",
    "E_emit_eV",
    "terminal_owner",
    "terminal_electrode",
    "KE_hit_eV",
    "steps",
]].head(30)

,electron_id,parent_id,generation,source_owner,source_electrode,emission_kind,E_emit_eV,terminal_owner,terminal_electrode,KE_hit_eV,steps
0,0,-1,1,sample,sample,BSE,68.773578,collector_shell,collector,117.418548,326


In [55]:
df_log = cascade_log_to_dataframe(cascade_log)
df_log

,electron_id,parent_id,generation,source_owner,source_electrode,source_Einc_eV,emission_kind,E_emit_eV,launch_offset_m,event,terminal_owner,terminal_electrode,terminal_Einc_eV,is_emitting_surface,N_child_emissions
0,0,-1,1,sample,sample,500.000025,BSE,68.773578,0.000126,NaN,NaN,NaN,NaN,NaN,NaN
1,0,-1,1,NaN,NaN,NaN,NaN,NaN,NaN,terminal_hit_for_possible_cascade,collector_shell,collector,117.418548,True,NaN
2,0,-1,1,NaN,NaN,NaN,NaN,NaN,NaN,child_emissions_sampled,collector_shell,collector,117.418548,NaN,0.0


In [64]:
primary_results_cas = []
cascade_results_all = []
cascade_logs_all = []

rng = np.random.default_rng(123)

for i in range(20):
    primary_cas_i, cas_i, log_i = run_one_primary_with_cascade(
        p_primary=p0s[i],
        v_primary=v0s[i],
        field=field,
        Ex_interp=Ex_interp,
        Ey_interp=Ey_interp,
        Ez_interp=Ez_interp,
        Phi_interp=Phi_interp,

        intersector_primary=intersector_primary,
        face_owner_primary=face_owner_primary,
        collision_mesh_primary=collision_mesh_primary,
        stl_boxes_primary=stl_boxes_primary,

        intersector_emit=intersector_emit,
        face_owner_emit=face_owner_emit,
        collision_mesh_emit=collision_mesh_emit,
        stl_boxes_emit=stl_boxes_emit,

        grid_transparency=grid_transparency,

        yield_models=yield_models,
        energy_models=energy_models,
        theta_models=theta_models,
        voltages=voltages,
        SEY_mult=1.0,
        rng=rng,

        sample_y_bounds=sample_y_bounds,
        sample_z_bounds=sample_z_bounds,

        max_generation=4,
        max_total_electrons=200,
        min_incident_energy_eV=0.5,

        emitted_max_step_fraction_of_h=0.75,
        emitted_dt_max=5.0e-11,
        launch_step_fraction_of_h=0.75,
    )

    for r in cas_i:
        r["primary_index"] = i

    for row in log_i:
        row["primary_index"] = i

    primary_results_cas.append(primary_cas_i)
    cascade_results_all.extend(cas_i)
    cascade_logs_all.extend(log_i)

df_cascade = cascade_results_to_dataframe(
    cascade_results_all,
    owner_name_map=field["owner_name_map"],
)

df_log = cascade_log_to_dataframe(cascade_logs_all)

print("N tracked cascade electrons:", len(df_cascade))

df_cascade["generation"].value_counts().sort_index()

N tracked cascade electrons: 40


generation
1    33
2     6
3     1
Name: count, dtype: int64

In [65]:
df_cascade["terminal_electrode"].value_counts()

terminal_electrode
collector    31
grid2         4
grid1         3
grid3         2
Name: count, dtype: int64

In [66]:
df_cascade["emission_kind"].value_counts()

emission_kind
SE     27
BSE    13
Name: count, dtype: int64

In [67]:
df_cascade[
    [
        "electron_id",
        "parent_id",
        "generation",
        "source_owner",
        "source_electrode",
        "emission_kind",
        "E_emit_eV",
        "terminal_owner",
        "terminal_electrode",
        "KE_hit_eV",
        "steps",
    ]
].head(50)

,electron_id,parent_id,generation,source_owner,source_electrode,emission_kind,E_emit_eV,terminal_owner,terminal_electrode,KE_hit_eV,steps
0,0,-1,1,sample,sample,BSE,68.773578,collector_shell,collector,117.414267,325
1,0,-1,1,sample,sample,BSE,435.732001,collector_shell,collector,483.203338,219
2,1,-1,1,sample,sample,SE,12.096402,collector_shell,collector,59.193227,727
3,2,-1,1,sample,sample,SE,11.267348,collector_shell,collector,60.383798,754
4,3,-1,1,sample,sample,SE,37.268456,collector_shell,collector,85.769386,428
5,4,2,2,collector_shell,collector,BSE,60.383798,collector_shell,collector,59.654946,703
6,0,-1,1,sample,sample,SE,11.213604,collector_shell,collector,58.072638,747
7,0,-1,1,sample,sample,SE,0.604176,collector_shell,collector,48.025258,3051
8,1,-1,1,sample,sample,SE,6.980049,collector_shell,collector,54.278978,935
9,0,-1,1,sample,sample,BSE,238.759042,g3_shell,grid3,238.759009,195


In [68]:
df_cascade["self_hit"] = (
    df_cascade["source_owner"].astype(str)
    == df_cascade["terminal_owner"].astype(str)
)

df_cascade.groupby(["source_owner", "terminal_owner"]).size().sort_values(ascending=False).head(20)

source_owner     terminal_owner 
sample           collector_shell    24
collector_shell  collector_shell     7
sample           g2_shell            4
                 g1frame             2
                 g3_shell            2
                 g1_shell            1
dtype: int64

In [69]:
df_cascade[df_cascade["self_hit"]][
    [
        "primary_index",
        "electron_id",
        "parent_id",
        "generation",
        "source_owner",
        "emission_kind",
        "E_emit_eV",
        "terminal_owner",
        "KE_hit_eV",
        "steps",
    ]
].sort_values("steps").head(20)

,primary_index,electron_id,parent_id,generation,source_owner,emission_kind,E_emit_eV,terminal_owner,KE_hit_eV,steps
22,12,3,2,3,collector_shell,SE,2.158931,collector_shell,3.259487,35
18,11,2,0,2,collector_shell,SE,13.420611,collector_shell,13.557440,40
20,12,1,0,2,collector_shell,SE,13.388958,collector_shell,13.777045,91
14,9,2,0,2,collector_shell,BSE,118.472653,collector_shell,119.853173,119
21,12,2,0,2,collector_shell,SE,22.636086,collector_shell,22.357915,128
35,17,3,0,2,collector_shell,SE,37.271668,collector_shell,37.985471,142
5,1,4,2,2,collector_shell,BSE,60.383798,collector_shell,59.654946,703


In [71]:
acct_cascade = summarize_cascade_accounting(
    cascade_results=cascade_results_all,
    N_primary=20,
    owner_name_map=field["owner_name_map"],
)

print_cascade_accounting_summary(acct_cascade)

acct_cascade["current_counts"]

Cascade accounting summary
--------------------------
N primary:              20
N cascade electrons:    40
N SE:                   27
N BSE:                  13
Max generation:         3
Per primary electrons:  2.00000

Electron-count balance by electrode:
           source_count  terminal_count  net_count
sample             33.0            -0.0       33.0
holder              0.0            -0.0        0.0
receiver            0.0            -0.0        0.0
rod                 0.0            -0.0        0.0
grid1               0.0            -3.0       -3.0
grid2               0.0            -4.0       -4.0
grid3               0.0            -2.0       -2.0
collector           7.0           -31.0      -24.0
drifttube           0.0            -0.0        0.0
escaped             0.0            -0.0        0.0
unknown             0.0            -0.0        0.0


,source_count,terminal_count,net_count
sample,33.0,-0.0,33.0
holder,0.0,-0.0,0.0
receiver,0.0,-0.0,0.0
rod,0.0,-0.0,0.0
grid1,0.0,-3.0,-3.0
grid2,0.0,-4.0,-4.0
grid3,0.0,-2.0,-2.0
collector,7.0,-31.0,-24.0
drifttube,0.0,-0.0,0.0
escaped,0.0,-0.0,0.0


In [72]:
acct_cascade["current_counts"]["net_count"].sum()

np.float64(0.0)

In [74]:
acct_cascade = add_per_primary_to_current_counts(acct_cascade)
acct_cascade["current_counts"]

,source_count,terminal_count,net_count,source_per_primary,terminal_per_primary,net_per_primary
sample,33.0,-0.0,33.0,1.65,-0.00,1.65
holder,0.0,-0.0,0.0,0.00,-0.00,0.00
receiver,0.0,-0.0,0.0,0.00,-0.00,0.00
rod,0.0,-0.0,0.0,0.00,-0.00,0.00
grid1,0.0,-3.0,-3.0,0.00,-0.15,-0.15
grid2,0.0,-4.0,-4.0,0.00,-0.20,-0.20
grid3,0.0,-2.0,-2.0,0.00,-0.10,-0.10
collector,7.0,-31.0,-24.0,0.35,-1.55,-1.20
drifttube,0.0,-0.0,0.0,0.00,-0.00,0.00
escaped,0.0,-0.0,0.0,0.00,-0.00,0.00


## Parallel cascades

In [10]:
grid_transparency = {
    "g1_shell": 0.93,
    "g2_shell": 0.93,
    "g3_shell": 0.93,
}

cascade = run_cascade_batch_parallel(
    N_primary=1000,
    E0_eV=500,
    field=field,
    Phi_interp=Phi_interp,
    Ex_interp=Ex_interp,
    Ey_interp=Ey_interp,
    Ez_interp=Ez_interp,

    intersector_primary=intersector_primary,
    face_owner_primary=face_owner_primary,
    collision_mesh_primary=collision_mesh_primary,
    stl_boxes_primary=stl_boxes_primary,

    intersector_emit=intersector_emit,
    face_owner_emit=face_owner_emit,
    collision_mesh_emit=collision_mesh_emit,
    stl_boxes_emit=stl_boxes_emit,

    grid_transparency=grid_transparency,

    yield_models=yield_models,
    energy_models=energy_models,
    theta_models=theta_models,
    voltages=voltages,

    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,

    grid_SEY_mult=1.0,
    collector_BSE_mult=1.0,

    max_generation=4,
    max_total_electrons_per_primary=200,
    min_incident_energy_eV=0.5,

    emitted_max_step_fraction_of_h=0.75,
    emitted_dt_max=5.0e-11,
    emitted_max_steps=20000,
    launch_step_fraction_of_h=0.75,

    seed=123,
    n_jobs=4,
    chunk_size=5,
    verbose=10,
)

print_cascade_batch_summary(cascade)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:  1.0min
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  1.4min
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:  2.5min
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  3.1min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  3.6min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:  4.2min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  4.7min
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:  5.3min
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:  6.1min
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:  6.8min
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:  7.8min
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:  8.5min
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  9.5min
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed: 10.7min
[Parallel(

Cascade batch summary
---------------------
N primary:               1000
N cascade electrons:     716
N SE:                    150
N BSE:                   566
Max generation:          4
Per primary electrons:   0.71600

Runtime:                 767.72 s
Runtime per primary:     0.7677 s
Launch failures:         0

Electron-count balance:
           source_count  terminal_count  net_count  source_per_primary  \
sample            489.0            -4.0      485.0               0.489   
holder              9.0           -12.0       -3.0               0.009   
receiver            4.0            -8.0       -4.0               0.004   
rod                 1.0            -4.0       -3.0               0.001   
grid1              10.0           -47.0      -37.0               0.010   
grid2              13.0           -45.0      -32.0               0.013   
grid3              14.0           -36.0      -22.0               0.014   
collector         172.0          -547.0     -375.0               0

In [11]:
paths = save_cascade_batch_tables(
    cascade,
    out_dir="results",
    prefix="cascade_cu_500eV_TEY_Tg=0p93",
)

paths

{'df_primary': WindowsPath('results/cascade_cu_500eV_TEY_Tg=0p93_primary.csv'),
 'df_cascade': WindowsPath('results/cascade_cu_500eV_TEY_Tg=0p93_cascade.csv'),
 'df_cascade_log': WindowsPath('results/cascade_cu_500eV_TEY_Tg=0p93_cascade_log.csv'),
 'df_grid_events': WindowsPath('results/cascade_cu_500eV_TEY_Tg=0p93_grid_events.csv'),
 'current_counts': WindowsPath('results/cascade_cu_500eV_TEY_Tg=0p93_current_counts.csv'),
 'summary': WindowsPath('results/cascade_cu_500eV_TEY_Tg=0p93_summary.csv')}

In [ ]:
result["df_cascade"]["reason"].value_counts()

In [12]:
df_log = cascade["df_cascade_log"]

grid_emissions = df_log[
    df_log["source_electrode"].isin(
        ["grid1", "grid2", "grid3"]
    )
    & df_log["event"].isin(
        ["tracked_emission", "launch_failed"]
    )
].copy()

grid_emissions["launch_status"] = np.where(
    grid_emissions["event"].eq("launch_failed"),
    "failed",
    "tracked",
)

counts = (
    grid_emissions
    .groupby([
        "source_electrode",
        "emission_kind",
        "emission_forward_backward",
        "launch_status",
    ])
    .size()
    .unstack(fill_value=0)
)

print(counts)

launch_status                                             failed  tracked
source_electrode emission_kind emission_forward_backward                 
grid1            BSE           backward                        0        2
                               forward                         0        1
                 SE            backward                        0        7
                               forward                         0        2
grid2            BSE           backward                        0        5
                               forward                         1        0
                 SE            backward                        0        9
                               forward                         0        2
grid3            BSE           backward                        0        6
                               forward                         0        1
                 SE            backward                        0        8
                               forward

In [13]:
angle_stats = (
    grid_emissions
    .groupby([
        "source_electrode",
        "emission_kind",
        "emission_forward_backward",
    ])["emission_scattering_angle_deg"]
    .describe()
)

print(angle_stats)

                                                          count        mean  \
source_electrode emission_kind emission_forward_backward                      
grid1            BSE           backward                     2.0  135.812226   
                               forward                      1.0   57.777721   
                 SE            backward                     7.0  135.059609   
                               forward                      2.0   51.793098   
grid2            BSE           backward                     5.0  121.192064   
                               forward                      1.0   53.835817   
                 SE            backward                     9.0  130.624394   
                               forward                      2.0   81.394937   
grid3            BSE           backward                     6.0  128.122013   
                               forward                      1.0   84.010753   
                 SE            backward             

In [14]:
cascade["df_cascade"].head()

,electron_id,parent_id,generation,source_owner,source_electrode,source_Einc_eV,emission_kind,E_emit_eV,launch_offset_m,emission_forward_cosine,...,reason,terminal_owner,terminal_electrode,owner_id,KE_hit_eV,steps,x_hit,y_hit,z_hit,primary_index
0,0,-1,1,sample,sample,499.601672,BSE,157.824061,0.000025,-0.392762,...,hit_fixed,collector_shell,collector,12.0,156.444882,442,0.022959,0.021701,-0.075683,4
1,0,-1,1,sample,sample,499.601506,BSE,497.100007,0.000025,-0.698438,...,hit_fixed,collector_shell,collector,12.0,495.847698,443,0.055991,0.047166,0.036948,5
2,0,-1,1,sample,sample,499.602101,BSE,499.147288,0.000025,-0.445742,...,hit_fixed,collector_shell,collector,12.0,498.458190,444,0.034299,-0.015416,-0.072998,11
3,1,0,2,collector_shell,collector,498.458190,SE,1.373314,0.000150,-0.946010,...,hit_fixed,collector_shell,collector,12.0,1.652428,40,0.033920,-0.015253,-0.073153,11
4,0,-1,1,sample,sample,499.601196,BSE,204.705228,0.000025,-0.785590,...,hit_fixed,collector_shell,collector,12.0,203.837662,445,0.061954,0.038598,-0.037624,12


In [90]:
cascade20["current_counts"]

,source_count,terminal_count,net_count,source_per_primary,terminal_per_primary,net_per_primary
sample,160.0,-0.0,160.0,1.60,-0.00,1.60
holder,1.0,-1.0,0.0,0.01,-0.01,0.00
receiver,0.0,-0.0,0.0,0.00,-0.00,0.00
rod,0.0,-0.0,0.0,0.00,-0.00,0.00
grid1,0.0,-17.0,-17.0,0.00,-0.17,-0.17
grid2,0.0,-19.0,-19.0,0.00,-0.19,-0.19
grid3,0.0,-14.0,-14.0,0.00,-0.14,-0.14
collector,59.0,-168.0,-109.0,0.59,-1.68,-1.09
drifttube,0.0,-0.0,0.0,0.00,-0.00,0.00
escaped,0.0,-1.0,-1.0,0.00,-0.01,-0.01


In [91]:
cascade20["df_cascade"]["generation"].value_counts().sort_index()

generation
1    160
2     29
3     25
4      6
Name: count, dtype: int64

In [80]:
paths = save_cascade_batch_tables(
    cascade20,
    out_dir="../results",
    prefix="cascade_500eV_N20_T09",
)

paths

{'df_primary': WindowsPath('../results/cascade_500eV_N20_T09_primary.csv'),
 'df_cascade': WindowsPath('../results/cascade_500eV_N20_T09_cascade.csv'),
 'df_cascade_log': WindowsPath('../results/cascade_500eV_N20_T09_cascade_log.csv'),
 'df_grid_events': WindowsPath('../results/cascade_500eV_N20_T09_grid_events.csv'),
 'current_counts': WindowsPath('../results/cascade_500eV_N20_T09_current_counts.csv'),
 'summary': WindowsPath('../results/cascade_500eV_N20_T09_summary.csv')}

In [92]:
seeds = [101, 102, 103, 104, 105]
rows = []

for seed in seeds:
    res = run_cascade_batch_parallel(
        N_primary=100,
        E0_eV=500,
        field=field,
        Phi_interp=Phi_interp,
        Ex_interp=Ex_interp,
        Ey_interp=Ey_interp,
        Ez_interp=Ez_interp,

        intersector_primary=intersector_primary,
        face_owner_primary=face_owner_primary,
        collision_mesh_primary=collision_mesh_primary,
        stl_boxes_primary=stl_boxes_primary,

        intersector_emit=intersector_emit,
        face_owner_emit=face_owner_emit,
        collision_mesh_emit=collision_mesh_emit,
        stl_boxes_emit=stl_boxes_emit,

        grid_transparency=grid_transparency,

        yield_models=yield_models,
        energy_models=energy_models,
        theta_models=theta_models,
        voltages=voltages,
        SEY_mult=1.0,

        sample_y_bounds=sample_y_bounds,
        sample_z_bounds=sample_z_bounds,

        max_generation=4,
        max_total_electrons_per_primary=200,
        min_incident_energy_eV=0.5,

        emitted_max_step_fraction_of_h=0.75,
        emitted_dt_max=5.0e-11,
        emitted_max_steps=20000,
        launch_step_fraction_of_h=0.75,

        seed=seed,
        n_jobs=4,
        chunk_size=5,
        verbose=0,
    )

    cc = res["current_counts"]

    row = {
        "seed": seed,
        "N_cascade_electrons": res["summary"]["N_cascade_electrons"],
        "max_generation": res["summary"]["max_generation"],
        "runtime_s": res["runtime_s"],
    }

    for electrode in ["sample", "grid1", "grid2", "grid3", "collector", "holder", "escaped"]:
        row[f"{electrode}_net_per_primary"] = cc.loc[electrode, "net_per_primary"]

    rows.append(row)

df_seed = pd.DataFrame(rows)
df_seed

,seed,N_cascade_electrons,max_generation,runtime_s,sample_net_per_primary,grid1_net_per_primary,grid2_net_per_primary,grid3_net_per_primary,collector_net_per_primary,holder_net_per_primary,escaped_net_per_primary
0,101,213,4,52.206100,1.67,-0.25,-0.15,-0.16,-1.09,0.00,-0.02
1,102,214,4,59.939268,1.70,-0.21,-0.14,-0.14,-1.17,0.00,-0.02
2,103,208,4,47.954948,1.66,-0.24,-0.16,-0.15,-1.09,0.01,-0.02
3,104,210,4,52.542635,1.62,-0.22,-0.13,-0.14,-1.10,0.00,-0.01
4,105,212,4,50.618575,1.63,-0.20,-0.16,-0.17,-1.08,0.00,-0.01


In [93]:
df_seed.describe()

,seed,N_cascade_electrons,max_generation,runtime_s,sample_net_per_primary,grid1_net_per_primary,grid2_net_per_primary,grid3_net_per_primary,collector_net_per_primary,holder_net_per_primary,escaped_net_per_primary
count,5.000000,5.000000,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,103.000000,211.400000,4.0,52.652305,1.656000,-0.224000,-0.148000,-0.152000,-1.106000,0.002000,-0.016000
std,1.581139,2.408319,0.0,4.458480,0.032094,0.020736,0.013038,0.013038,0.036469,0.004472,0.005477
min,101.000000,208.000000,4.0,47.954948,1.620000,-0.250000,-0.160000,-0.170000,-1.170000,0.000000,-0.020000
25%,102.000000,210.000000,4.0,50.618575,1.630000,-0.240000,-0.160000,-0.160000,-1.100000,0.000000,-0.020000
50%,103.000000,212.000000,4.0,52.206100,1.660000,-0.220000,-0.150000,-0.150000,-1.090000,0.000000,-0.020000
75%,104.000000,213.000000,4.0,52.542635,1.670000,-0.210000,-0.140000,-0.140000,-1.090000,0.000000,-0.010000
max,105.000000,214.000000,4.0,59.939268,1.700000,-0.200000,-0.130000,-0.140000,-1.080000,0.010000,-0.010000


In [86]:
def compare_firstgen_terminal_and_cascade_net(firstgen_result, cascade_result):
    """
    Compare first-generation terminal arrival counts with cascade net balance.

    Note:
        first_generation_arrivals are positive terminal arrival counts.
        cascade_net_balance is signed source-minus-arrival electron balance.
    """
    rows = []

    fg_counts = firstgen_result["electrode_counts"]

    for electrode, count in fg_counts.items():
        rows.append({
            "model": "first_generation_arrivals_positive",
            "electrode": electrode,
            "count": count,
            "per_primary": count / firstgen_result["summary"]["N_primary"],
        })

    cc = cascade_result["current_counts"]

    for electrode in cc.index:
        rows.append({
            "model": "cascade_net_source_minus_arrival",
            "electrode": electrode,
            "count": cc.loc[electrode, "net_count"],
            "per_primary": cc.loc[electrode, "net_per_primary"],
        })

    return pd.DataFrame(rows)

In [82]:
df_compare_fc = compare_firstgen_and_cascade(
    batch20_parallel,
    cascade20,
)

df_compare_fc

,model,electrode,count,per_primary
0,first_generation_terminal_only,sample,0.0,0.00
1,first_generation_terminal_only,holder,0.0,0.00
2,first_generation_terminal_only,receiver,0.0,0.00
3,first_generation_terminal_only,rod,0.0,0.00
4,first_generation_terminal_only,grid1,1.0,0.05
5,first_generation_terminal_only,grid2,2.0,0.10
6,first_generation_terminal_only,grid3,3.0,0.15
7,first_generation_terminal_only,collector,23.0,1.15
8,first_generation_terminal_only,drifttube,0.0,0.00
9,first_generation_terminal_only,escaped,0.0,0.00


In [83]:
def add_cascade_step_diagnostics(result: dict) -> dict:
    """
    Add step diagnostics for cascade trajectories.
    """
    df = result["df_cascade"]

    if df.empty:
        result["step_diagnostics"] = {}
        return result

    diag = {
        "N_electrons": int(len(df)),
        "steps_mean": float(df["steps"].mean()),
        "steps_median": float(df["steps"].median()),
        "steps_max": int(df["steps"].max()),
        "steps_p90": float(df["steps"].quantile(0.90)),
        "steps_p95": float(df["steps"].quantile(0.95)),
        "steps_p99": float(df["steps"].quantile(0.99)),
        "slowest_electrons": df.sort_values("steps", ascending=False).head(10),
    }

    result["step_diagnostics"] = diag

    return result


def print_cascade_step_diagnostics(result: dict):
    """
    Print cascade step diagnostics.
    """
    if "step_diagnostics" not in result:
        result = add_cascade_step_diagnostics(result)

    d = result["step_diagnostics"]

    if not d:
        print("No cascade electrons.")
        return

    print("Cascade step diagnostics")
    print("------------------------")
    print(f"N electrons:   {d['N_electrons']}")
    print(f"steps mean:    {d['steps_mean']:.1f}")
    print(f"steps median:  {d['steps_median']:.1f}")
    print(f"steps p90:     {d['steps_p90']:.1f}")
    print(f"steps p95:     {d['steps_p95']:.1f}")
    print(f"steps p99:     {d['steps_p99']:.1f}")
    print(f"steps max:     {d['steps_max']}")

    display_cols = [
        "primary_index",
        "electron_id",
        "parent_id",
        "generation",
        "source_owner",
        "emission_kind",
        "E_emit_eV",
        "terminal_owner",
        "terminal_electrode",
        "KE_hit_eV",
        "steps",
    ]

    print("\nSlowest cascade electrons:")
    print(d["slowest_electrons"][display_cols])

In [84]:
cascade20 = add_cascade_step_diagnostics(cascade20)
print_cascade_step_diagnostics(cascade20)

Cascade step diagnostics
------------------------
N electrons:   32
steps mean:    605.5
steps median:  453.0
steps p90:     1275.0
steps p95:     1495.0
steps p99:     2152.1
steps max:     2371

Slowest cascade electrons:
    primary_index  electron_id  parent_id  generation source_owner  \
4               1            2         -1           1       sample   
28             17            2         -1           1       sample   
13              6            0         -1           1       sample   
3               1            1         -1           1       sample   
14              6            1         -1           1       sample   
15              7            0         -1           1       sample   
23             12            0         -1           1       sample   
1               0            1         -1           1       sample   
25             16            0         -1           1       sample   
21             11            1         -1           1       sample   

   em

## Accounting

In [13]:
# ============================================================
# Simple current-sign convention
# ============================================================

def simple_terminal_charge_counts(df_emit: pd.DataFrame) -> pd.Series:
    """
    Simple electron-arrival counts per electrode.

    Positive number means number of electrons arriving at that electrode.
    This is not yet the electrical conventional current sign.

    For electrical current on an electrode, multiply electron arrival rate by:

        I = -e * arrival_rate

    But for yield/current-distribution debugging, raw electron counts are easier.
    """
    return count_by_electrode(df_emit)


def electron_arrival_current_A(
    df_emit: pd.DataFrame,
    primary_rate_per_s: float,
    N_primary: int,
    e_charge_C: float = 1.602176634e-19,
) -> pd.Series:
    """
    Convert electron arrival counts to conventional electrode current.

    Electrons arriving on an electrode give negative conventional current
    into that electrode:

        I = -e * arrival_rate

    Parameters
    ----------
    df_emit:
        Emitted-electron dataframe.
    primary_rate_per_s:
        Number of primary electrons per second.
    N_primary:
        Number of simulated primary electrons represented by df_emit.
    """
    counts = count_by_electrode(df_emit)

    arrival_rate = counts / N_primary * primary_rate_per_s

    return -e_charge_C * arrival_rate

In [16]:
acct = summarize_first_generation(
    primary_result=primary_res,
    emitted_results=emitted,
    N_primary=1,
    owner_name_map=field["owner_name_map"],
)

print_first_generation_summary(acct)

acct["df_emit"]

First-generation sample-emission accounting
------------------------------------------
N primary:       1
Primary sample hits: 1
N emitted total: 2
N SE:            1
N BSE:           1
N quantum refl.: 0

Terminal electrodes:
terminal_electrode
sample       0
holder       0
receiver     0
rod          0
grid1        2
grid2        0
grid3        0
collector    0
drifttube    0
escaped      0
unknown      0
Name: count, dtype: int64

Terminal owners:
terminal_owner
g1_shell    2
Name: count, dtype: int64

Emission kinds:
emission_kind
BSE    1
SE     1
Name: count, dtype: int64


,electron_index,emission_kind,E_emit_eV,primary_E_inc_eV,primary_cos_theta,reason,terminal_owner,terminal_electrode,owner_id,KE_hit_eV,steps,x_hit,y_hit,z_hit
0,0,BSE,199.094207,500.000025,1.0,hit_fixed,g1_shell,grid1,9,199.094161,264,0.025665,-0.030761,0.019911
1,1,SE,9.945011,500.000025,1.0,hit_fixed,g1_shell,grid1,9,9.944952,1197,0.022262,0.033053,-0.020539


In [17]:
electron_arrival_current_A(
    acct["df_emit"],
    primary_rate_per_s=1.0e9,
    N_primary=1,
)

terminal_electrode
sample      -0.000000e+00
holder      -0.000000e+00
receiver    -0.000000e+00
rod         -0.000000e+00
grid1       -3.204353e-10
grid2       -0.000000e+00
grid3       -0.000000e+00
collector   -0.000000e+00
drifttube   -0.000000e+00
escaped     -0.000000e+00
unknown     -0.000000e+00
Name: count, dtype: float64